In [1]:
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server

In [2]:
def get_htmx_idx(hdrs):
    return next((i for i, hdr in enumerate(app.hdrs) if (hdr.attrs.get('src') or '').endswith('htmx.min.js')), -1)

In [3]:
from fasthtml.common import *
import asyncio
from datetime import datetime

In [4]:
app, rt = fast_app()

In [5]:
app.hdrs

[meta((),{'charset': 'utf-8'}),
 meta((),{'name': 'viewport', 'content': 'width=device-width, initial-scale=1, viewport-fit=cover'}),
 script((),{'src': 'https://cdn.jsdelivr.net/npm/htmx.org@2.0.6/dist/htmx.min.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js'}),
 (link((),{'rel': 'stylesheet', 'href': 'https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css'}),
  style((':root { --pico-font-size: 100%; }',),{})),
 script(("\n    function sendmsg() {\n        window.parent.postMessage({height: document.documentElement.offsetHeight}, '*');\n    }\n    window.onload = function() {\n        sendmsg();\n        document.body.addEventListener('htmx:afterSettle',    sendmsg);\n        document.body.addEventListener('htmx:wsAfterMessage', sendmsg);\n    };",)

In [6]:
htmx_idx = get_htmx_idx(app.hdrs)
app.hdrs.insert(htmx_idx+1, Script(src="https://unpkg.com/htmx-ext-sse@2.2.3/dist/sse.min.js"))

In [7]:
app.hdrs

[meta((),{'charset': 'utf-8'}),
 meta((),{'name': 'viewport', 'content': 'width=device-width, initial-scale=1, viewport-fit=cover'}),
 script((),{'src': 'https://cdn.jsdelivr.net/npm/htmx.org@2.0.6/dist/htmx.min.js'}),
 script(('',),{'src': 'https://unpkg.com/htmx-ext-sse@2.2.3/dist/sse.min.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js'}),
 script((),{'src': 'https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js'}),
 (link((),{'rel': 'stylesheet', 'href': 'https://cdn.jsdelivr.net/npm/@picocss/pico@latest/css/pico.min.css'}),
  style((':root { --pico-font-size: 100%; }',),{})),
 script(("\n    function sendmsg() {\n        window.parent.postMessage({height: document.documentElement.offsetHeight}, '*');\n    }\n    window.onload = function() {\n        sendmsg();\n        document.body.addEventListener('htmx:afterSettle',    sendmsg);\n  

In [8]:
from fasthtml.components import hx_attrs_annotations
hx_attrs_annotations

{'hx_swap': typing.Union[typing.Literal['innerHTML', 'outerHTML', 'afterbegin', 'beforebegin', 'beforeend', 'afterend', 'delete', 'none'], str, NoneType],
 'hx_swap_oob': typing.Union[typing.Literal['true', 'innerHTML', 'outerHTML', 'afterbegin', 'beforebegin', 'beforeend', 'afterend', 'delete', 'none'], str, NoneType],
 'hx_push_url': typing.Union[typing.Literal['true', 'false'], str, NoneType],
 'hx_replace_url': typing.Union[typing.Literal['true', 'false'], str, NoneType],
 'hx_disabled_elt': typing.Union[typing.Literal['this', 'next', 'previous'], str, NoneType],
 'hx_history': typing.Union[typing.Literal['false'], str, NoneType],
 'hx_params': typing.Union[typing.Literal['*', 'none'], str, NoneType],
 'hx_validate': typing.Optional[typing.Literal['true', 'false']],
 'hx_post': typing.Optional[str],
 'hx_headers': typing.Optional[str],
 'hx_disable': typing.Optional[str],
 'hx_request': typing.Optional[str],
 'hx_select': typing.Optional[str],
 'hx_sync': typing.Optional[str],
 'hx

In [9]:
import random
from asyncio import sleep
from fasthtml.common import *

In [10]:
@rt
def index():
    return Titled("SSE Random Number Generator",
        P("Generate pairs of random numbers, as the list grows scroll downwards."),
        Div(hx_ext="sse",
            sse_connect="/number-stream",
            hx_swap="beforeend show:bottom",
            sse_swap="message"))

shutdown_event = signal_shutdown()

async def number_generator():
    while not shutdown_event.is_set():
        data = Article(random.randint(1, 100))
        yield sse_message(data)
        await sleep(1)

@rt("/number-stream")
async def get(): return EventStream(number_generator())

In [11]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX())

In [12]:
# Stop server when done
server.stop()